In [ ]:
# deepseek下载模型路径："C:\Users\Administrator\.ollama\models\manifests\registry.ollama.ai\library\deepseek-r1\1.5b"

In [5]:
from ollama import chat

def annotate(prompt):
    response = chat(
        model='deepseek-r1:1.5b',
        messages=[{'role': 'user', 'content': prompt}])
    return response.message.content.split("</think>\n\n")[-1]

annotate("hello")

'Hello! How can I assist you today? 😊'

In [ ]:
# 设备设置
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [1]:
import csv
import ollama
import numpy as np
from datasets import load_dataset
from tqdm import tqdm
import time

C:\Users\Administrator\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 配置参数
class Config:
    # 数据集参数
    dataset_name = "ag_news"
    text_field = "text"  # 数据集新闻文本字段
    label_field = "label" # 数据集新闻标签字段
    label_names = ["World", "Sports", "Business", "Science/Tech"]
    
    # 模型参数
    ollama_model = "deepseek-r1:1.5b"
    max_retries = 3                   # API调用重试次数
    retry_delay = 2                   # 重试等待时间（秒）
    
    # 输出路径
    output_csv = "data/ag_news_labels.csv"

cfg = Config()

In [3]:
classification_prompt = """As an expert news analyst, provide probability distribution for this news text:

**Text**: {text}

**Categories**:
1. World - International affairs, politics, disasters
2. Sports - Athletic competitions, players, match results
3. Business - Financial markets, corporate activities
4. Science/Tech - Technological innovations, scientific discoveries

**Requirements**:
1. Analyze text content
2. List key entities
3. Output FOUR probabilities (space-separated) summing to 1.0

**Output Format**:
prob_world prob_sports prob_business prob_sci_tech

Example 1:
Text: "Apple launches new AI-powered chip"
Output: 0.02 0.0 0.28 0.70

Example 2:
Text: "Eurozone leaders reach debt agreement"
Output: 0.85 0.0 0.15 0.0

Now provide probabilities for this text:"""

In [ ]:
# 增强的解析函数
import re

def parse_probabilities(raw_output):
    try:
        # 去除非数字字符
        cleaned = re.sub(r'[^\d\s\.]', '', raw_output)
        
        # 逆序查找最后出现的4个数值
        lines = cleaned.strip().split('\n')
        for line in reversed(lines): #倒叙寻找
            parts = line.strip().split()
            if len(parts) == 4:
                # 验证是否为合法概率值
                probs = list(map(float, parts))
                if all(0 <= p <= 1 for p in probs):
                    # 强制归一化
                    total = sum(probs)
                    if abs(total - 1.0) > 0.001:
                        probs = [p/total for p in probs]
                    return [round(p, 4) for p in probs]
        
        # 未找到有效数值时返回均匀分布
        return [0.25]*4
    except:
        return [0.25]*4

In [5]:
full_prompt = classification_prompt.format(text="Today is a terribe day for bosses.Trump added 60% tax for China.")
response = ollama.chat(model=cfg.ollama_model,
                        messages=[{'role': 'user', 'content': full_prompt}],
                        options={'temperature': 0.0}
                        )
raw_output = response['message']['content']
probs = parse_probabilities(raw_output)
print(raw_output)
print(probs)

<think>
Alright, let's tackle this probability distribution problem step by step. The task is to analyze the given news text and assign probabilities to it belonging to each of the four categories provided.

First, I need to understand what the text says. It mentions that "Today is a terribe day for bosses. Trump added 60% tax for China." So, the key entities here are "bosses" and "Trump adding 60% tax for China."

Now, looking at the categories:

1. **World - International affairs, politics, disasters**: This category includes events related to international politics or disasters. The text discusses a boss day event and a political addition (tax), so it definitely falls under World.

2. **Sports - Athletic competitions, players, match results**: There's no mention of sports here. It's about business-related topics, so Sports probability is 0.

3. **Business - Financial markets, corporate activities**: The text discusses tax additions, which are part of corporate financial matters. So,

In [6]:
import csv
import pandas as pd
dataset=pd.read_csv(r"data\ag_news_test.csv")
print(dataset[:2])

   label                                              title  \
0      3                  Fears for T N pension after talks   
1      4  The Race is On: Second Private Team Sets Launc...   

                                                text  
0  Unions representing workers at Turner   Newall...  
1  SPACE.com - TORONTO, Canada -- A second\team o...  


In [ ]:
# 数据处理流程
def generate_labels(dataset):
    # 创建CSV文件
    with open(cfg.output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        header = ["text", "hard_label"] + cfg.label_names
        writer.writerow(header)
    
    # 分批处理
    for i in tqdm(range(0, 100, 5), desc="处理中========="):
        batch = dataset[i:i+5]
        batch_rows = []
            
        for text, hard_label in zip(batch["text"], batch["label"]):
            # 生成提示词
            full_prompt = classification_prompt.format(text=text)
            #print("已生成提示词！============")
                
            # 调用模型
            for _ in range(cfg.max_retries):
                try:
                    response = ollama.chat(
                        model=cfg.ollama_model,
                        messages=[{'role': 'user', 'content': full_prompt}],
                        options={'temperature': 0.0}
                    )
                    raw_output = response['message']['content']
                    probs = parse_probabilities(raw_output)
                    break
                except:
                    probs = [0.25]*4
                    time.sleep(cfg.retry_delay)
                
            # 构建数据行
            batch_rows.append([text, hard_label] + probs)
            #print("已完成调用！============")
            
        # 批量写入
        with open(cfg.output_csv, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerows(batch_rows)

In [ ]:
if __name__ == "__main__":
    generate_labels(dataset)
    print(f"标注数据已保存至：{cfg.output_csv}")

处理中=========:  25%|██▌       | 5/20 [08:11<22:59, 91.95s/it] 